In [1]:
import os
import sys
import json
import time
import re
import pandas as pd

from datetime import datetime

from glob import glob

# append ../src to sys.path
sys.path.append("../")

from src.processing.helper_processing.visualization_event import (
    plot_event_syncing,
    render_event,
)

from src.core.game_event import GameEvent
from src.core.game import Game

### read random events (or not so random)

In [2]:
# Filter for relevant events
relevant_event_types = [
    "score_change",
    "shot_off_target",
    "shot_blocked",
    "shot_saved",
    "seven_m_missed",
]

In [1]:
import os
import re
import glob
import json
from IPython.display import clear_output

# Create dictionaries to map match_id to file paths
sportradar_match_mapping = {}
kinexon_match_mapping = {}

specific_match_id = None
specific_name_team_home = None

# Example file paths for matches (not events)
list_of_match_files_sportradar = glob.glob("../data/raw/gameday_*/sportradar/*.json")
list_of_match_files_kinexon = glob.glob("../data/raw/gameday_*/kinexon/*.csv")

# Example file paths for events (used later to load event-level data)
list_of_event_files_sportradar = glob.glob("../data/events/match_*/*.json")
list_of_event_files_kinexon = glob.glob("../data/events/match_*/*.csv")

# Normalize paths to handle different OS path separators
list_of_match_files_sportradar = [os.path.normpath(file) for file in list_of_match_files_sportradar]
list_of_match_files_kinexon = [os.path.normpath(file) for file in list_of_match_files_kinexon]
list_of_event_files_sportradar = [os.path.normpath(file) for file in list_of_event_files_sportradar]
list_of_event_files_kinexon = [os.path.normpath(file) for file in list_of_event_files_kinexon]

# Regex for extracting match_id from filenames
match_regex = r"_id_(\d+)_"
event_regex = r"match_(\d+)[\\/]+event_(\d+)"

# Map match_id to the match-level file paths (for sportradar and kinexon)
for file in list_of_match_files_sportradar:
    match = re.search(match_regex, file)
    if match:
        match_id = match.group(1)
        sportradar_match_mapping[match_id] = file

for file in list_of_match_files_kinexon:
    match = re.search(match_regex, file)
    if match:
        match_id = match.group(1)
        kinexon_match_mapping[match_id] = file

# Map match_id and event_id to the event file paths
sportradar_mapping = {}
kinexon_mapping = {}
for file in list_of_event_files_sportradar:
    match = re.search(event_regex, file)
    if match:
        match_id = match.group(1)
        event_id = match.group(2)
        sportradar_mapping[(match_id, event_id)] = file

for file in list_of_event_files_kinexon:
    match = re.search(event_regex, file)
    if match:
        match_id = match.group(1)
        event_id = match.group(2)
        kinexon_mapping[(match_id, event_id)] = file

counter = 0

# logging level to warn
import logging
logging.basicConfig(level=logging.WARN)

# Iterate over the match files, ensuring both sportradar and kinexon data exist for the same match_id
for match_id in sportradar_match_mapping.keys():
    if match_id in kinexon_match_mapping:
        sportradar_match_file = sportradar_match_mapping[match_id]
        kinexon_match_file = kinexon_match_mapping[match_id]


        if not "TVBStuttgart_v" in sportradar_match_file:
            continue
        
        print(f"Processing match_id: {match_id} for files: {sportradar_match_file} and {kinexon_match_file}")

        # Check if specific match_id is set
        if specific_match_id is not None and specific_match_id not in sportradar_match_file:
            print(f"Skipping match_id: {match_id} because it does not match the specific match_id: {specific_match_id}")
            continue

        # Iterate over the event files for the current match
        for (event_match_id, event_id) in sportradar_mapping.keys():
            if event_match_id == match_id and (event_match_id, event_id) in kinexon_mapping:
                sportradar_event_file = sportradar_mapping[(event_match_id, event_id)]
                kinexon_event_file = kinexon_mapping[(event_match_id, event_id)]

                # print match id
                print(f"MATCH ID: {match_id}")
                print(f"Processing sportradar event file: {sportradar_event_file} and kinexon event file: {kinexon_event_file}")

                # Load the event data from Sportradar
                with open(sportradar_event_file, 'r') as f:
                    event_data = json.load(f)

                # Check if the event type is relevant
                if event_data["type"] not in relevant_event_types:
                    continue

                # Check if specific team name is set in the event file
                if specific_name_team_home is not None and specific_name_team_home not in event_data["name_team_home"]:
                    continue

                # switch attack direction if specific team is playing home
                if specific_name_team_home is not None:
                    if specific_name_team_home in event_data["name_team_home"]:
                        event_data["attack_direction"] = "right" if event_data["attack_direction"] == "left" else "left"
                    else:
                        event_data["attack_direction"] = "left" if event_data["attack_direction"] == "right" else "right"

                # Initialize the GameEvent class with event data and kinexon path
                game_event = GameEvent(
                    event=event_data,
                    path_to_kinexon_scene=kinexon_event_file
                )
                # check if event_time_throw is in the event class
                # if "event_time_throw" in game_event.to_dict():
                #     if game_event.to_dict()["event_time_throw"] is not None:
                #         continue

                # Clear cell output to avoid clutter
                clear_output()

                # again for debugging
                game_event = GameEvent(
                    event=event_data,
                    path_to_kinexon_scene=kinexon_event_file
                )
                
                # Clear cell output to avoid clutter
                clear_output()

                
                # Perform additional operations like syncing and rendering
                plot_event_syncing(game_event)
                render_event(game_event)

                clear_output()
                # Clear output to avoid clutter
                # clear_output()

                counter += 1
                print(f"Processed {counter} events.")

                if counter > 1:
                    break

                


Processing match_id: 42307447 for files: ..\data\raw\gameday_02\sportradar\2023-09-01_gd_02_id_42307447_teams_TVBStuttgart_vs_FuchseBerlin_sportradar.json and ..\data\raw\gameday_02\kinexon\2023-09-01_gd_02_id_42307447_teams_TVBStuttgart_vs_FuchseBerlin_kinexon.csv
MATCH ID: 42307447
Processing sportradar event file: ..\data\events\match_42307447\event_1531729613_sportradar.json and kinexon event file: ..\data\events\match_42307447\event_1531729613_positions.csv


NameError: name 'relevant_event_types' is not defined

In [ ]:
# import os
# import re
# import glob
# import json
# from IPython.display import clear_output

# # Create dictionaries to map match_id to file paths
# sportradar_match_mapping = {}
# kinexon_match_mapping = {}

# specific_match_id = None
# specific_name_team_home = None

# # Example file paths for matches (not events)
# list_of_match_files_sportradar = glob.glob("../data/raw/gameday_*/sportradar/*.json")
# list_of_match_files_kinexon = glob.glob("../data/raw/gameday_*/kinexon/*.csv")

# # Example file paths for events (used later to load event-level data)
# list_of_event_files_sportradar = glob.glob("../data/events/match_*/*.json")
# list_of_event_files_kinexon = glob.glob("../data/events/match_*/*.csv")

# # Normalize paths to handle different OS path separators
# list_of_match_files_sportradar = [os.path.normpath(file) for file in list_of_match_files_sportradar]
# list_of_match_files_kinexon = [os.path.normpath(file) for file in list_of_match_files_kinexon]
# list_of_event_files_sportradar = [os.path.normpath(file) for file in list_of_event_files_sportradar]
# list_of_event_files_kinexon = [os.path.normpath(file) for file in list_of_event_files_kinexon]

# # Regex for extracting match_id from filenames
# match_regex = r"_id_(\d+)_"
# event_regex = r"match_(\d+)[\\/]+event_(\d+)"

# # Map match_id to the match-level file paths (for sportradar and kinexon)
# for file in list_of_match_files_sportradar:
#     match = re.search(match_regex, file)
#     if match:
#         match_id = match.group(1)
#         sportradar_match_mapping[match_id] = file

# for file in list_of_match_files_kinexon:
#     match = re.search(match_regex, file)
#     if match:
#         match_id = match.group(1)
#         kinexon_match_mapping[match_id] = file

# # Map match_id and event_id to the event file paths
# sportradar_mapping = {}
# kinexon_mapping = {}
# for file in list_of_event_files_sportradar:
#     match = re.search(event_regex, file)
#     if match:
#         match_id = match.group(1)
#         event_id = match.group(2)
#         sportradar_mapping[(match_id, event_id)] = file

# for file in list_of_event_files_kinexon:
#     match = re.search(event_regex, file)
#     if match:
#         match_id = match.group(1)
#         event_id = match.group(2)
#         kinexon_mapping[(match_id, event_id)] = file

# counter = 0

# # Iterate over the match files, ensuring both sportradar and kinexon data exist for the same match_id
# for match_id in sportradar_match_mapping.keys():
#     if match_id in kinexon_match_mapping:
#         sportradar_match_file = sportradar_match_mapping[match_id]
#         kinexon_match_file = kinexon_match_mapping[match_id]

#         print(f"Processing match_id: {match_id} for files: {sportradar_match_file} and {kinexon_match_file}")

#         if not "Flensburg" in sportradar_match_file:
#             continue

#         # Check if specific match_id is set
#         if specific_match_id is not None and specific_match_id not in sportradar_match_file:
#             print(f"Skipping match_id: {match_id} because it does not match the specific match_id: {specific_match_id}")
#             continue

#         # Initialize the Game object with match-level files
#         game = Game(path_file_sportradar=sportradar_match_file, path_file_kinexon=kinexon_match_file)

        
#         # Filter by specific team name if set
#         if specific_name_team_home is not None:
#             if specific_name_team_home not in game.dict_sportradar["sport_event"]["competitors"][0]["name"]:
#                 print(f"Skipping match_id: {match_id} because it does not match the specific team name: {specific_name_team_home} but is: {game.dict_sportradar['sport_event']['competitors'][0]['name']}")
#                 continue

#         print(f'Processing match_id: {match_id} for files: {sportradar_match_file} and {kinexon_match_file}')
        
        

#         # process game events
#         for (
#             event_id,
#             dict_event,
#         ) in game.dict_kinexon_path_by_event_id.items():
#             # create game event object
#             if dict_event["type"] not in relevant_event_types:
#                 continue

#             # Check if specific team name is set in the event file
#             if specific_name_team_home is not None and specific_name_team_home not in dict_event["name_team_home"]:
#                     continue
            
#             # switch attack direction if Stuttgart is playing home
#             if specific_name_team_home is not None:
#                 if specific_name_team_home in dict_event["name_team_home"]:
#                     dict_event["attack_direction"] = "right" if dict_event["attack_direction"] == "left" else "left"
#                 else:
#                     dict_event["attack_direction"] = "left" if dict_event["attack_direction"] == "left" else "right"

#             game_event = GameEvent(
#                 dict_event,
#                 game.dict_kinexon_path_by_event_id[event_id][
#                     "path_kinexon"
#                 ],
#             )
#             plot_event_syncing(game_event)
#             render_event(game_event)
            
#             # Clear output to avoid clutter
#             clear_output()

#             counter += 1
#             print(f"Processed {counter} events.")

#             if counter > 2:
#                 break

